# 학생 모델 QLoRA 학습

**실행 전 준비물** (Google Drive `MyDrive/ade-project/`에 업로드):
- `unified.jsonl` (항상 필요)
- `teacher_outputs.jsonl` (R2/R3/R4 조건에만 필요 — generate_teacher_data.ipynb의 출력)
- `cadec_v1_sct.jsonl` (R1 조건에만 필요 — 로컬 `research-project/data/cadec_v1_sct.jsonl`)

`fold_assignment.csv`는 git에 커밋되어 있어 clone 시 자동으로 딸려온다.

**주의**: R2/R3/R4 생성(generate_teacher_data.ipynb)과 같은 세션에서 돌리지 말 것 (VRAM 부족).

In [5]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ade-project'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
REPO_URL = 'https://github.com/Gaeul5/Oracle_healthcare-bio_sLLM.git'

import os
if not os.path.exists('/content/repo'):
    !git clone -q {REPO_URL} /content/repo
else:
    !git -C /content/repo pull -q
%cd /content/repo/research-project

/content/repo/research-project


In [4]:
!pip install -q -U transformers accelerate bitsandbytes peft datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 72.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 22.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.2 MB/s eta 0:00:00:00:0100:01


## 실험 설정

이 셀만 바꿔서 조건/모델 크기/축을 조합해 재실행하면 된다 (30개 조합을 셀마다 따로 만드는 대신 여기 값만 바꿔가며 반복 실행).

- `CONDITION`: `R0` / `R1` / `R2` / `R3` / `R4` (R1은 CADEC v1만, R2-R4는 teacher_outputs.jsonl 필요)
- 주 축을 쓰려면 `TRAIN_DOMAIN`을 `'forum'` 또는 `'literature'`로, `CV_ROUND`는 `None`
- 보조 축(포럼 내부 4-fold)을 쓰려면 `TRAIN_DOMAIN=None`, `CV_ROUND`를 1~4로 (R1은 1 또는 4만 가능 — cadec_v1이 fold 2라 2,3은 학습 예시 0건)

In [11]:
MODEL_SIZE = '0.5b'        # '0.5b' | '1.5b' | '3b'
CONDITION = 'R0'           # 'R0' | 'R1' | 'R2' | 'R3' | 'R4'
TRAIN_DOMAIN = 'forum'     # 'forum' | 'literature' | None
CV_ROUND = None            # 1 | 2 | 3 | 4 | None
SEED = 0

assert (TRAIN_DOMAIN is None) != (CV_ROUND is None), 'TRAIN_DOMAIN과 CV_ROUND 중 정확히 하나만 지정'

axis_tag = TRAIN_DOMAIN if TRAIN_DOMAIN else f'cvround{CV_ROUND}'
run_name = f'{MODEL_SIZE}_{CONDITION}_{axis_tag}_seed{SEED}'

cmd = (
    f'python src/train_qlora.py '
    f'--unified {DRIVE_DIR}/unified.jsonl '
    f'--teacher-outputs {DRIVE_DIR}/teacher_outputs.jsonl '
    f'--cadec-v1-sct {DRIVE_DIR}/cadec_v1_sct.jsonl '
    f'--fold-assignment data/fold_assignment.csv '
    f'--model-size {MODEL_SIZE} --condition {CONDITION} '
    + (f'--train-domain {TRAIN_DOMAIN} ' if TRAIN_DOMAIN else f'--cv-round {CV_ROUND} ')
    + f'--seed {SEED} '
    f'--out-dir {DRIVE_DIR}/adapters/{run_name}'
)
print(cmd)

python src/train_qlora.py --unified /content/drive/MyDrive/ade-project/unified.jsonl --teacher-outputs /content/drive/MyDrive/ade-project/teacher_outputs.jsonl --cadec-v1-sct /content/drive/MyDrive/ade-project/cadec_v1_sct.jsonl --fold-assignment data/fold_assignment.csv --model-size 0.5b --condition R0 --train-domain forum --seed 0 --out-dir /content/drive/MyDrive/ade-project/adapters/0.5b_R0_forum_seed0


In [ ]:
!{cmd}

## 주 축 자동 순차 실행 (forum ↔ literature)

위 두 셀(설정값 수동 변경 → 재실행)을 조합마다 반복하는 대신, 주 축에 필요한
전체 조합을 아래 셀 하나로 순차 실행한다. 범위는 사전등록된 H1/H2 검증에
필요한 조합만: `0.5b/1.5b/3b × R0/R2/R3/R4 × forum/literature × seed 0/1/2`
= 72회 (R1과 cv_round 보조 축은 별도 — 여기 포함 안 함).

- 이미 완료된 조합(`out_dir`에 `adapter_config.json` 존재)은 자동으로 건너뛴다 →
  Colab 세션이 끊겨도 이 셀을 다시 실행하면 중단된 지점부터 이어간다.
- 조합 하나가 실패해도 전체를 멈추지 않고 다음 조합으로 넘어간다. 실패 목록은
  마지막에 출력되고, `run_log.jsonl`에 조합별 성공/실패·소요시간이 남는다.
- 72회 전부 완료하려면 세션 여러 개에 걸쳐 이 셀을 여러 번 실행해야 할 가능성이 높다.

In [ ]:
import itertools
import json
import os
import subprocess
import time

MODEL_SIZES = ['0.5b', '1.5b', '3b']
CONDITIONS = ['R0', 'R2', 'R3', 'R4']
TRAIN_DOMAINS = ['forum', 'literature']
SEEDS = [0, 1, 2]

combos = list(itertools.product(MODEL_SIZES, CONDITIONS, TRAIN_DOMAINS, SEEDS))

adapters_dir = f'{DRIVE_DIR}/adapters'
logs_dir = f'{adapters_dir}/logs'
os.makedirs(adapters_dir, exist_ok=True)
os.makedirs(logs_dir, exist_ok=True)
log_path = f'{adapters_dir}/run_log.jsonl'


def run_and_stream(cmd, tail_n=40):
    """자식 프로세스 출력을 한 줄씩 직접 print()로 흘려보낸다.

    subprocess.run()이 fd를 그대로 상속시키는 방식은 이 노트북 환경에서
    출력이 씹히는 문제가 있었다 (전체 프로세스가 실패해도 traceback이
    전혀 안 보임) — Popen + PIPE로 매 줄을 명시적으로 print해서 우회한다.
    """
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    tail = []
    for line in proc.stdout:
        print(line, end='')
        tail.append(line)
        if len(tail) > tail_n:
            tail.pop(0)
    proc.wait()
    return proc.returncode, ''.join(tail)


failed = []
skipped = 0
ran = 0

for i, (model_size, condition, train_domain, seed) in enumerate(combos, 1):
    run_name = f'{model_size}_{condition}_{train_domain}_seed{seed}'
    out_dir = f'{adapters_dir}/{run_name}'

    if os.path.exists(f'{out_dir}/adapter_config.json'):
        print(f'[{i}/{len(combos)}] {run_name} — 이미 완료, 건너뜀')
        skipped += 1
        continue

    cmd = (
        f'python src/train_qlora.py '
        f'--unified {DRIVE_DIR}/unified.jsonl '
        f'--teacher-outputs {DRIVE_DIR}/teacher_outputs.jsonl '
        f'--model-size {model_size} --condition {condition} '
        f'--train-domain {train_domain} '
        f'--seed {seed} '
        f'--out-dir {out_dir}'
    )

    print(f'[{i}/{len(combos)}] {run_name} 시작...')
    t0 = time.time()
    returncode, tail = run_and_stream(cmd)
    elapsed = time.time() - t0
    ran += 1

    with open(f'{logs_dir}/{run_name}.log', 'w', encoding='utf-8') as f:
        f.write(tail)

    status = 'success' if returncode == 0 else 'failed'
    if status == 'failed':
        failed.append(run_name)
        print(f'--- {run_name} 실패, 마지막 출력 (전체 로그: logs/{run_name}.log) ---')
        print(tail)
        print('--- 여기까지 ---')

    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(json.dumps({
            'run_name': run_name,
            'status': status,
            'returncode': returncode,
            'elapsed_sec': round(elapsed, 1),
        }, ensure_ascii=False) + '\n')

    print(f'[{i}/{len(combos)}] {run_name} — {status} ({elapsed:.0f}s)')

print(f'\n종료. 이번 세션에서 실행 {ran}건 / 건너뜀 {skipped}건 / 남은 조합 없음: {ran + skipped == len(combos)}')
if failed:
    print(f'실패한 조합 ({len(failed)}건): {failed}')
else:
    print('실패 없음.')

[1/72] 0.5b_R0_forum_seed0 시작...
학습 예시 5516건 (조건=R0, 도메인=forum)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 491.74it/s]
trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359

Map: 100%|██████████| 5516/5516 [00:05<00:00, 1035.54 examples/s]

  2%|▏         | 20/1035 [00:43<36:44,  2.17s/it]
                                                 

  4%|▍         | 40/1035 [01:27<37:01,  2.23s/it]
                                                 

  6%|▌         | 60/1035 [02:10<35:40,  2.20s/it]
                                                 

  8%|▊         | 80/1035 [02:54<34:19,  2.16s/it]
                                                 

 10%|▉         | 100/1035 [03:41<34:49,  2.23s/it]
                                                  

 12%|█▏        | 120/1035 [04:27<35:59,  2.36s/it]
                                                  

 14%|█▎        | 140/1035 [05:14<34:22,  2.30s/it]
                                                  

 15%|█▌  